In [0]:

CREATE OR REPLACE FUNCTION cineai.app.get_ticket(
  p_ticket_id STRING COMMENT 'Exact ticket ID, for example TKT000205'
)
RETURNS STRING
LANGUAGE SQL
COMMENT 'Purpose: Retrieve operational information for one ticket without personal data. Args: p_ticket_id is an exact ticket identifier. Returns: JSON containing ticket status, screening ID, sales channel, purchase timestamp, amount and payment method. Failure: Returns INVALID_TICKET_ID or TICKET_NOT_FOUND. Authorization: Customer ownership must be verified separately.'
RETURN
SELECT CASE
  WHEN p_ticket_id IS NULL
    OR NOT (p_ticket_id RLIKE '^TKT[0-9]{6}$')
  THEN '{"error":"INVALID_TICKET_ID"}'
  ELSE COALESCE(
    (
      SELECT TO_JSON(
        NAMED_STRUCT(
          'ticket_id', t.ticket_id,
          'screening_id', t.screening_id,
          'ticket_status', t.status,
          'sales_channel', t.sales_channel,
          'purchase_timestamp', t.purchase_timestamp,
          'total_amount_eur', t.total_amount_eur,
          'payment_method', t.payment_method
        )
      )
      FROM cineai.raw.ticket_sales AS t
      WHERE t.ticket_id = p_ticket_id
      LIMIT 1
    ),
    '{"error":"TICKET_NOT_FOUND"}'
  )
END;
